# Adaptive RAG：答案有來源，就代表答對了嗎？

入口先選本地或外部；出口分開檢查「來源支持」與「回答原問題」。測試會刻意插入一個假的結論，再插入一個有證據卻答非所問的結論。

這是課程新寫的可重現實驗；資料為虛構餐廳示例。預設使用真正的 LangGraph、稀疏向量索引和可檢查的固定策略，不使用模型或真實搜尋服務。第一格需網路安裝套件，之後執行不需要金鑰。

## 1. 準備環境
Colab：選「執行階段 → 全部執行」。本機：依 README 建立 Python 3.12 環境。

In [ ]:
# Colab 第一次執行會安裝課程固定版本；本機已按 README 安裝時不連網。
from importlib.metadata import version, PackageNotFoundError
import sys, subprocess
try:
    installed = version("langgraph")
except PackageNotFoundError:
    installed = None
if installed != "1.2.20":
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "langgraph==1.2.20"])
from langgraph.graph import StateGraph, START, END
from typing import TypedDict, Annotated
from operator import add
from math import sqrt

## 2. 檢查可以引用的資料
只有本地菜單、營業時間與一筆外部預報樣本。先找出你預期回答會引用的來源 ID。

In [ ]:
# 小型稀疏向量索引：維度是詞彙；向量餘弦相似度決定排序，不呼叫 embedding API。
LOCAL = [
    {"id":"menu", "kind":"menu", "tags":["菜單","素食","餐點"], "text":"素食菜單有絲瓜蒸餃。"},
    {"id":"hours", "kind":"hours", "tags":["營業","時間"], "text":"教學餐廳的營業時間為 11:00–20:00。"},
]
WEB = [{"id":"web-demo", "kind":"weather", "tags":["天氣","下雨"], "text":"課堂固定預報樣本：有雨。"}]
REFUSAL = "目前沒有足夠來源回答。"
VOCAB = sorted({tag for doc in LOCAL for tag in doc["tags"]})
def vector(query):
    return [float(tag in query) for tag in VOCAB]
def cosine(x, y):
    return sum(a*b for a,b in zip(x,y)) / (sqrt(sum(a*a for a in x)*sum(b*b for b in y)) or 1)
def search_local(query):
    scored = [(cosine(vector(query),vector(" ".join(d["tags"]))),d) for d in LOCAL]
    score, doc = max(scored, key=lambda pair: pair[0])
    return [doc] if score > 0 else []
def intent(question):
    for doc in LOCAL + WEB:
        if any(tag in question for tag in doc["tags"]):
            return doc["kind"]
    return "hours" if "打烊" in question else "unknown"
def rewritten(question):
    return {"hours":"營業時間","menu":"素食菜單","weather":"天氣樣本"}.get(intent(question), question)
def cite(doc):
    return doc["text"] + " [" + doc["id"] + "]"

class State(TypedDict, total=False):
    question: str
    query: str
    docs: list
    answer: str
    route: str
    relevant: bool
    verdict: str
    retries: int
    generations: int
    fault: str
    trace: Annotated[list[str], add]

def step(name, **values):
    return {"trace":[name], **values}
def retrieve(s):
    return step("Retrieve", docs=search_local(s["query"]))
def grade(s):
    ok = any(d["kind"] == intent(s["question"]) for d in s["docs"])
    return step("grade_documents", relevant=ok)
def transform(s):
    return step("transform_query", query=rewritten(s["question"]), retries=s.get("retries",0)+1)
def generate(s):
    doc = next((d for d in s["docs"] if d["kind"]==intent(s["question"])), None)
    return step("Generate", answer=cite(doc) if doc else REFUSAL, generations=s.get("generations",0)+1)
def abstain(s):
    return step("Abstain", answer=REFUSAL)
def web_fixture(s):
    # 固定外部結果，不會真的搜尋網路；未知主題一律回空。
    return step("Web Search (fixture)", docs=[d for d in WEB if any(t in s["query"] for t in d["tags"])])

def run(question, **kw):
    out=graph.invoke({"question":question,"query":question,**kw},{"recursion_limit":30})
    print(" → ".join(out["trace"]), "\n", out["answer"])
    return out

## 3. 把投影片的方塊和箭頭接成圖
每個節點回傳部分 state；每條條件邊決定下一步。追 trace 觀察實際走法。

In [ ]:
def route(s):
    dest="search" if intent(s["question"])=="weather" else "retrieve"
    return step("RouteQuery",route=dest)
def draft(s):
    out=generate(s)
    # 可控制的一次性壞答案：用來驗證出口真的能擋下錯誤。
    if out["generations"]==1 and s.get("fault")=="unsupported":
        out["answer"]="整間餐廳 24 小時營業。 [hours]"
    if out["generations"]==1 and s.get("fault")=="offtopic":
        out["answer"]=cite(LOCAL[0])
    return out
def grade_answer(s):
    # 支持度：本練習只允許精確引文；不是看到 [來源] 就通過。
    evidence = next((d for d in s["docs"] if cite(d)==s["answer"]),None)
    if not evidence: verdict="unsupported"
    elif evidence["kind"] != intent(s["question"]): verdict="not_useful"
    else: verdict="useful"
    return step("GradeHallucinations / GradeAnswer: "+verdict, verdict=verdict)
def after_grade(s):
    if s["verdict"]=="useful": return END
    if s.get("generations",0)>=3: return "stop"
    if s["verdict"]=="unsupported": return "draft"
    return "rewrite" if s.get("retries",0)<1 else "stop"
def retrieve_context(s):
    out=retrieve(s)
    # 假設檢索有雜訊：核心文件旁夾入相關餐廳的菜單。回答可能有根據、卻答非所問。
    if s.get("fault")=="offtopic" and out["docs"] and LOCAL[0] not in out["docs"]:
        out["docs"] += [LOCAL[0]]
    return out

g=StateGraph(State)
for n,fn in {"route":route,"retrieve":retrieve_context,"grade":grade,"rewrite":transform,
             "search":web_fixture,"draft":draft,"check":grade_answer,"stop":abstain}.items():
    g.add_node(n,fn)
g.add_edge(START,"route")
g.add_conditional_edges("route",lambda s:s["route"],["retrieve","search"])
g.add_edge("retrieve","grade")
g.add_conditional_edges("grade",lambda s:"draft" if s["relevant"] else
                        ("rewrite" if s.get("retries",0)<1 else "stop"),["draft","rewrite","stop"])
g.add_edge("rewrite","route"); g.add_edge("search","draft"); g.add_edge("draft","check")
g.add_conditional_edges("check",after_grade,[END,"draft","rewrite","stop"]); g.add_edge("stop",END)
graph=g.compile()

## 4. 執行驗收
斷言失敗表示實際分支或答案與規格不符。

In [ ]:
a=run("今天會下雨嗎？")
assert a["trace"][1]=="Web Search (fixture)" and "Retrieve" not in a["trace"] and a["answer"]==cite(WEB[0])
b=run("營業時間？",fault="unsupported")
assert b["generations"]==2 and b["answer"]==cite(LOCAL[1]) and not b.get("retries",0)
c=run("營業時間？",fault="offtopic")
assert c["retries"]==1 and c["generations"]==2 and c["answer"]==cite(LOCAL[1])
assert any(t.endswith("not_useful") for t in c["trace"])
assert run("停車費多少？")["answer"]==REFUSAL

## 5. 自己動手
比較 b 和 c：哪個案例只需重寫答案，哪個案例必須重新檢索？把 b 的幻覺字串改成另一句有 [hours] 卻未被文件支持的話，確認驗收仍擋得住。

下一步：需要測真實模型時，依資料夾 README 的「Claude Agent SDK」執行獨立的 live graph；不會影響這本的無金鑰驗收。